# Pattern 4: Gateway + Cedar Policy Engine

Add Cedar authorization policies to the Gateway. Cedar evaluates whether a principal
can invoke a specific Gateway target. Supports `LOG_ONLY` (audit) and `ENFORCE` (block) modes.

**What you get:** Per-principal authorization, audit logging, policy-as-code.

**What Cedar controls:** Whether a principal can invoke a Gateway (resource-level).

**What Cedar doesn't control:** Which documents are returned (use metadata filters for that).

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents, and creates the KB execution role. (The setup cell here re-runs it
  idempotently, and additionally creates the **Gateway role**.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control` — Gateway, Policy
  Engine, Cedar policies), S3, and IAM.

## Architecture

```
Agent (IAM) ──► Gateway ──► Cedar Policy Engine ──► KB Target ──► Managed KB
                  │              │
                  │              └── permit/deny based on principal + resource
                  └── LOG_ONLY: evaluate but don't block
                      ENFORCE:  evaluate and block if denied
```

In [ ]:
import boto3
import time
import json
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent),
# then create the Gateway role the AgentCore Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,
    region_name=REGION,
)
ROLE_ARN    = info["role_arn"]
S3_BUCKET   = info["bucket"]
S3_PREFIX   = info["prefix"]
GW_ROLE_ARN = util.create_gateway_role(region_name=REGION)

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
S3_ACCOUNT = session.client("sts").get_caller_identity()["Account"]

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")


In [ ]:
# Step 1: Create Policy Engine
pe_response = ac.create_policy_engine(
    name=f"p4_pe_{int(time.time())}"
)

pe_id = pe_response["policyEngineId"]
pe_arn = pe_response["policyEngineArn"]
print(f"Policy Engine: {pe_id}")
print(f"Policy Engine ARN: {pe_arn}")

# Wait for ACTIVE
for _ in range(12):
    pe = ac.get_policy_engine(policyEngineId=pe_id)
    if pe["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Status: {pe['status']}")

In [ ]:
# Step 2: Create KB + Data Source + Ingest
response = cp.create_knowledge_base(
    name=f"p4-cedar-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)
print(f"Ingestion: {job['status']}")

In [ ]:
# Step 3: Create Gateway with Policy Engine (LOG_ONLY mode)
gw_response = ac.create_gateway(
    name=f"p4-cedar-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="AWS_IAM",
    policyEngineConfiguration={
        "arn": pe_arn,
        "mode": "LOG_ONLY"  # Evaluate policies but don't block requests
    }
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

# Wait for READY
gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Step 4: Create Cedar policy — permit at gateway ARN level
# Build the gateway ARN for the Cedar resource
gw_arn = f"arn:aws:bedrock-agentcore:{REGION}:{S3_ACCOUNT}:gateway/{gw_id}"

cedar_statement = f'permit(principal, action, resource == AgentCore::Gateway::"{gw_arn}");'
print(f"Cedar policy:\n  {cedar_statement}")

# Policy names must be unique across policy engines, so derive one from gw_id
# (create_policy is not idempotent — a static name collides on re-runs).
policy_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"permit_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": cedar_statement}},
    validationMode="IGNORE_ALL_FINDINGS",
)
policy_id = policy_response["policyId"]
print(f"Policy: {policy_id} [{policy_response['status']}]")

# Wait for ACTIVE
for _ in range(12):
    p = ac.get_policy(policyEngineId=pe_id, policyId=policy_id)
    if p["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Policy status: {p['status']}")


In [ ]:
# Step 5: Create KB Target
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — this is what
                    # the LLM reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {
                                "numberOfResults": 5
                            }
                        }
                    }
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")

In [ ]:
# Step 6: Retrieve THROUGH the gateway using the official MCP client (SigV4-signed).
# This is the real gateway path — unlike dp.retrieve(), which hits Bedrock
# directly by kb_id and bypasses the gateway (so Cedar never sees it).
#
# An AWS_IAM gateway requires every request to be SigV4-signed. The MCP client
# talks over httpx, which has no built-in SigV4, so we wrap botocore's signer in
# a small httpx.Auth and hand it to the client via `http_client=`.
import httpx
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.exceptions import McpError


class SigV4HTTPXAuth(httpx.Auth):
    """httpx auth handler that SigV4-signs each request for the bedrock-agentcore service."""
    requires_request_body = True   # we must see the body to sign it

    def __init__(self, credentials, service, region):
        self._credentials, self._service, self._region = credentials, service, region

    def auth_flow(self, request):
        aws_req = AWSRequest(
            method=request.method,
            url=str(request.url),
            data=request.content,
            headers=dict(request.headers),
        )
        SigV4Auth(self._credentials, self._service, self._region).add_auth(aws_req)
        request.headers.update(dict(aws_req.headers))   # copy the signed headers back
        yield request


sigv4 = SigV4HTTPXAuth(session.get_credentials(), "bedrock-agentcore", REGION)

# The forbid policy in Step 7 also hides the tool from tools/list, so we resolve
# the tool name here (while permitted) and reuse it in Step 8. Naming convention
# is `<target-name>___<tool-name>`, so we can also build it directly.
tool_name = "kb-retrieve___Retrieve"

async with httpx.AsyncClient(auth=sigv4) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session_mcp:
            await session_mcp.initialize()               # MCP handshake (required)

            # tools/list — what an agent sees over MCP (permitted in LOG_ONLY mode).
            listing = await session_mcp.list_tools()
            tool_name = next(t.name for t in listing.tools if t.name.split("___")[-1] == "Retrieve")
            print(f"Gateway tool: {tool_name}")

            # tools/call — retrieve through the gateway (KB ID never sent by us).
            result = await session_mcp.call_tool(
                name=tool_name,
                arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
            )
            print(f"isError: {result.isError}")
            print("=== Retrieved via gateway ===")
            print(result.content[0].text[:800] if result.content else result)

## Cedar Policy Format

Cedar policies use the `AgentCore::Gateway` entity type — the resource is the Gateway ARN.
Two things matter for a policy that actually **enforces** (both verified live):

- the gateway must be in **`ENFORCE`** mode (`LOG_ONLY` evaluates + logs but never blocks), and
- the **resource must be constrained** to a specific `AgentCore::Gateway` — a bare wildcard
  resource is rejected at create time.

```cedar
// Permit all callers to invoke this gateway
permit(principal, action, resource == AgentCore::Gateway::"<gateway-arn>");

// Forbid all callers on this gateway (overrides the permit — this is what Step 7 does)
forbid(principal, action, resource == AgentCore::Gateway::"<gateway-arn>");
```

## LOG_ONLY vs ENFORCE

| Mode | Behavior |
|---|---|
| `LOG_ONLY` | Cedar evaluates the policy and logs the decision, but **does not block** requests. Use for auditing and testing policies before enforcement. |
| `ENFORCE` | Cedar evaluates and **blocks** denied requests (the caller gets a JSON-RPC `-32002` error). Use after validating in LOG_ONLY. |

## What Cedar Controls vs What It Doesn't

| Cedar controls | Cedar does NOT control |
|---|---|
| Whether callers can invoke a Gateway | Which documents are returned |
| Resource-level permit/deny | Document-level filtering |
| Gateway-level access | Metadata filter scoping |

For document-level scoping, combine Cedar (Pattern 4) with metadata filters (Pattern 2).



In [ ]:
# Step 7: Enforce a Cedar FORBID on this gateway.
# Two things are required for a real block (both verified):
#   1. gateway mode = ENFORCE (LOG_ONLY only logs, never blocks)
#   2. forbid scoped to the gateway RESOURCE (a bare wildcard resource is rejected)
# forbid overrides the earlier permit, so ALL callers are denied on this gateway.

# 1. Flip the gateway from LOG_ONLY to ENFORCE (re-send its current config).
gw_info = ac.get_gateway(gatewayIdentifier=gw_id)
ac.update_gateway(
    gatewayIdentifier=gw_id,
    name=gw_info["name"],
    roleArn=gw_info["roleArn"],
    protocolType="MCP",
    authorizerType="AWS_IAM",
    policyEngineConfiguration={"arn": pe_arn, "mode": "ENFORCE"},
)
for _ in range(24):
    if ac.get_gateway(gatewayIdentifier=gw_id)["status"] == "READY":
        break
    time.sleep(5)
print("Gateway mode: ENFORCE")

# 2. Add the forbid policy (unique name per gateway to avoid re-run collisions).
forbid_statement = (
    f'forbid(\n'
    f'  principal,\n'
    f'  action,\n'
    f'  resource == AgentCore::Gateway::"{gw_arn}"\n'
    f');'
)
print(f"Cedar forbid policy:\n{forbid_statement}\n")

deny_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"deny_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": forbid_statement}},
    validationMode="IGNORE_ALL_FINDINGS",
)
deny_policy_id = deny_response["policyId"]
for _ in range(12):
    st = ac.get_policy(policyEngineId=pe_id, policyId=deny_policy_id)["status"]
    if st in ("ACTIVE", "CREATE_FAILED"):
        break
    time.sleep(5)
print(f"Deny policy: {deny_policy_id} [{st}]")
time.sleep(15)  # let the decision propagate to the gateway


In [ ]:
# Step 8: Retrieve through the gateway again — now BLOCKED by Cedar.
# Same MCP client as Step 6, but ENFORCE + forbid means the gateway denies the
# tool call and raises an McpError with JSON-RPC code -32002. We catch it INSIDE
# the session context, where it surfaces directly — once it unwinds through the
# client's task group on context exit it gets wrapped in nested ExceptionGroups,
# which are awkward to unpack. The forbid also hides the tool from tools/list
# (it now returns []), so we call the tool by the name resolved in Step 6.
from mcp.shared.exceptions import McpError

async with httpx.AsyncClient(auth=sigv4) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session_mcp:
            await session_mcp.initialize()
            try:
                result = await session_mcp.call_tool(
                    name=tool_name,
                    arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
                )
                print("Unexpected — call was allowed:")
                print(result.content[0].text[:400] if result.content else result)
            except McpError as e:
                if e.error.code == -32002:
                    print("BLOCKED by Cedar policy enforcement")
                    print(f"   {e.error.message}")
                else:
                    print(f"Unexpected MCP error [{e.error.code}]: {e.error.message}")

In [ ]:
# Cleanup — order: policies → target → gateway → policy engine → DS → KB
ac.delete_policy(policyEngineId=pe_id, policyId=policy_id)
ac.delete_policy(policyEngineId=pe_id, policyId=deny_policy_id)
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
time.sleep(3)
ac.delete_policy_engine(policyEngineId=pe_id)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Policy Engine {pe_id}, Gateway {gw_id}, KB {kb_id}")
